# RoBERTa Frame Classification Training 

**Model**: RoBERTa-base 

**Optimizations Applied**:
1. Extended training (15 epochs)
2. Cosine LR schedule with warmup
3. Larger effective batch size (32)
4. Macro F1 optimization
5. Label smoothing (0.1)
6. Early stopping (patience=4)
7. Confidence threshold tuning

In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TRANSFORMERS_NO_TF'] = '1'

import pickle
import numpy as np
import torch
import json
from datasets import load_from_disk
from transformers import (
    RobertaForSequenceClassification,
    RobertaTokenizer,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB)")

PyTorch: 2.5.1+cu121
CUDA: True
GPU: NVIDIA GeForce GTX 1060 (6.4GB)


## 1. Load Configuration & Datasets

In [2]:
# Load RoBERTa config
with open('data/roberta_config.pkl', 'rb') as f:
    config = pickle.load(f)

with open('data/roberta_label_encoder.pkl', 'rb') as f:
    label_encoder = pickle.load(f)

print("Configuration:")
for k, v in config.items():
    print(f"  {k}: {v}")
print(f"\nLabels: {list(label_encoder.classes_)}")

Configuration:
  model_name: roberta-base
  max_length: 384
  num_labels: 6
  label_mapping: {0: 'Conflict', 1: 'Economic', 2: 'Human Impact', 3: 'Moral Value', 4: 'None', 5: 'Powerlessness'}
  train_size: 1638
  val_size: 360
  test_size: 348
  group_aware_split: True
  split_random_state: 42

Labels: ['Conflict', 'Economic', 'Human Impact', 'Moral Value', 'None', 'Powerlessness']


In [3]:
# Load tokenized datasets (with validation split)
print("\nLoading datasets...")
tokenized_dataset = load_from_disk('data/roberta_tokenized_data')
print(tokenized_dataset)
print(f"\nTrain: {len(tokenized_dataset['train']):,}")
print(f"Val: {len(tokenized_dataset['validation']):,}")
print(f"Test: {len(tokenized_dataset['test']):,}")


Loading datasets...
DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 1638
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 360
    })
    test: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 348
    })
})

Train: 1,638
Val: 360
Test: 348


## 2. Compute Class Weights

In [4]:
train_labels = np.array(tokenized_dataset['train']['label'])
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

print("✅ Class Weights:")
print("="*60)
for idx, (label, weight) in enumerate(zip(label_encoder.classes_, class_weights)):
    count = (train_labels == idx).sum()
    print(f"  {label:20s}: weight={weight:.4f} (n={count})")
print("="*60)

✅ Class Weights:
  Conflict            : weight=1.0000 (n=273)
  Economic            : weight=1.0000 (n=273)
  Human Impact        : weight=1.0000 (n=273)
  Moral Value         : weight=1.0000 (n=273)
  None                : weight=1.0000 (n=273)
  Powerlessness       : weight=1.0000 (n=273)


## 3. Weighted Loss Trainer

In [5]:
class WeightedLossTrainer(Trainer):
    """Custom Trainer with class-weighted loss"""
    
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        if class_weights is not None:
            self.class_weights = class_weights.to(self.args.device)
        else:
            self.class_weights = None
    
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        """Weighted cross-entropy loss"""
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        
        if self.class_weights is not None:
            loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights)
            loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        else:
            loss = outputs.loss
        
        return (loss, outputs) if return_outputs else loss

print("✅ Weighted Loss Trainer defined")

✅ Weighted Loss Trainer defined


## 4. Load Model (DAPT or Vanilla RoBERTa)

In [6]:
import os.path

# Try DAPT model first, fall back to vanilla
if os.path.exists('models/roberta-brexit-dapt'):
    MODEL_PATH = 'models/roberta-brexit-dapt'
    print("✅ Loading DOMAIN-ADAPTED RoBERTa (DAPT)")
else:
    MODEL_PATH = 'roberta-base'
    print("⚠️  DAPT model not found, using vanilla roberta-base")
    print("   (Run 03a_domain_adaptive_pretraining.ipynb for +5-10% F1)")

tokenizer = RobertaTokenizer.from_pretrained(MODEL_PATH)
model = RobertaForSequenceClassification.from_pretrained(
    MODEL_PATH,
    num_labels=config['num_labels'],
    id2label=config['label_mapping'],
    label2id={v: k for k, v in config['label_mapping'].items()},
    problem_type='single_label_classification'
)

print(f"\n✅ Model: {MODEL_PATH}")
print(f"   Parameters: {model.num_parameters():,}")
print(f"   Labels: {config['num_labels']}")

⚠️  DAPT model not found, using vanilla roberta-base
   (Run 03a_domain_adaptive_pretraining.ipynb for +5-10% F1)


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



✅ Model: roberta-base
   Parameters: 124,650,246
   Labels: 6


## 5. Enhanced Metrics Function

In [7]:
def compute_metrics(eval_pred):
    """Enhanced metrics with per-class tracking"""
    predictions_output = eval_pred.predictions
    labels = eval_pred.label_ids
    
    if isinstance(predictions_output, tuple):
        logits = predictions_output[0]
    else:
        logits = predictions_output
    
    predictions = np.argmax(logits, axis=-1)
    
    accuracy = accuracy_score(labels, predictions)
    f1_macro = f1_score(labels, predictions, average='macro')
    f1_weighted = f1_score(labels, predictions, average='weighted')
    
    precision, recall, f1, support = precision_recall_fscore_support(
        labels, predictions, average=None, zero_division=0
    )
    
    metrics = {
        'accuracy': float(accuracy),
        'f1_macro': float(f1_macro),
        'f1_weighted': float(f1_weighted),
    }
    
    for idx, label_name in config['label_mapping'].items():
        metrics[f'f1_{label_name}'] = float(f1[idx])
    
    print(f"\n📊 Per-class F1:")
    for idx, label_name in config['label_mapping'].items():
        print(f"  {label_name:20s}: {f1[idx]:.4f} (n={int(support[idx])})")
    print(f"  {'MACRO':20s}: {f1_macro:.4f}")
    
    return metrics

print("✅ Metrics function defined")

✅ Metrics function defined


## 6. Training Arguments

In [8]:
training_args = TrainingArguments(
    output_dir='models/roberta-optimized',
    
    # Extended training
    num_train_epochs=15,
    
    # Cosine LR + warmup
    learning_rate=3e-5,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    
    # Effective batch 32 (RoBERTa lighter than BART)
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    
    weight_decay=0.01,
    
    # Macro F1 optimization
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    
    # Label smoothing
    label_smoothing_factor=0.1,
    
    # Logging
    logging_dir='results/logs_roberta',
    logging_steps=25,
    report_to=['tensorboard'],
    
    # Performance
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=0,
    seed=42,
)

print("✅ Training Configuration:")
print("="*70)
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  LR: {training_args.learning_rate} ({training_args.lr_scheduler_type})")
print(f"  Warmup: {training_args.warmup_ratio}")
print(f"  Batch: {training_args.per_device_train_batch_size} x {training_args.gradient_accumulation_steps} = {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Metric: {training_args.metric_for_best_model}")
print(f"  Label smoothing: {training_args.label_smoothing_factor}")
print(f"  FP16: {training_args.fp16}")
print("="*70)

✅ Training Configuration:
  Epochs: 15
  LR: 3e-05 (SchedulerType.COSINE)
  Warmup: 0.1
  Batch: 16 x 2 = 32
  Metric: f1_macro
  Label smoothing: 0.1
  FP16: True


## 7. Initialize Trainer

In [9]:
trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    compute_metrics=compute_metrics,
    class_weights=class_weights_tensor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)]
)

print("✅ Trainer initialized")

✅ Trainer initialized


## 8. Train Model

In [10]:
print("\n🚀 Training...")
print("="*80)
print("Target: Macro F1 > 0.70")
print("="*80)

train_result = trainer.train()

print("\n✅ Training complete!")
print(f"Time: {train_result.metrics['train_runtime']:.1f}s ({train_result.metrics['train_runtime']/60:.1f}min)")


🚀 Training...
Target: Macro F1 > 0.70


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted,F1 Conflict,F1 Economic,F1 Human impact,F1 Moral value,F1 None,F1 Powerlessness
1,1.774500,1.614500,0.480556,0.453374,0.453374,0.395349,0.597561,0.436782,0.420290,0.591195,0.279070
2,1.127100,0.981366,0.633333,0.616550,0.616550,0.423529,0.796992,0.736842,0.583333,0.741935,0.416667
3,0.732500,0.891115,0.655556,0.650577,0.650577,0.606897,0.781250,0.702703,0.527273,0.780488,0.504854
4,0.480900,1.016796,0.638889,0.634661,0.634661,0.461538,0.774194,0.727273,0.536585,0.742857,0.565517
5,0.265500,1.129647,0.650000,0.644900,0.644900,0.554455,0.796992,0.734375,0.524138,0.754386,0.505051
6,0.162300,1.233052,0.655556,0.658035,0.658035,0.551724,0.769231,0.736000,0.545455,0.742857,0.602941
7,0.091100,1.422374,0.652778,0.647917,0.647917,0.600000,0.803150,0.708661,0.442308,0.778761,0.554622
8,0.028000,1.774604,0.625000,0.627950,0.627950,0.555556,0.783333,0.672000,0.495726,0.718447,0.542636
9,0.014300,1.845764,0.647222,0.645511,0.645511,0.602941,0.765217,0.705882,0.475248,0.759259,0.564516
10,0.008200,1.864551,0.672222,0.663896,0.663896,0.562500,0.793893,0.703125,0.509091,0.784000,0.630769



📊 Per-class F1:
  Conflict            : 0.3953 (n=60)
  Economic            : 0.5976 (n=60)
  Human Impact        : 0.4368 (n=60)
  Moral Value         : 0.4203 (n=60)
  None                : 0.5912 (n=60)
  Powerlessness       : 0.2791 (n=60)
  MACRO               : 0.4534

📊 Per-class F1:
  Conflict            : 0.4235 (n=60)
  Economic            : 0.7970 (n=60)
  Human Impact        : 0.7368 (n=60)
  Moral Value         : 0.5833 (n=60)
  None                : 0.7419 (n=60)
  Powerlessness       : 0.4167 (n=60)
  MACRO               : 0.6165

📊 Per-class F1:
  Conflict            : 0.6069 (n=60)
  Economic            : 0.7812 (n=60)
  Human Impact        : 0.7027 (n=60)
  Moral Value         : 0.5273 (n=60)
  None                : 0.7805 (n=60)
  Powerlessness       : 0.5049 (n=60)
  MACRO               : 0.6506

📊 Per-class F1:
  Conflict            : 0.4615 (n=60)
  Economic            : 0.7742 (n=60)
  Human Impact        : 0.7273 (n=60)
  Moral Value         : 0.5366 (n=60)
  N

## 9. Validation Evaluation

In [11]:
print("\n📊 Validation evaluation...")
val_results = trainer.evaluate(eval_dataset=tokenized_dataset['validation'])

print(f"\nAccuracy: {val_results['eval_accuracy']:.4f}")
print(f"Macro F1: {val_results['eval_f1_macro']:.4f}")


📊 Validation evaluation...



📊 Per-class F1:
  Conflict            : 0.5625 (n=60)
  Economic            : 0.7939 (n=60)
  Human Impact        : 0.7031 (n=60)
  Moral Value         : 0.5091 (n=60)
  None                : 0.7840 (n=60)
  Powerlessness       : 0.6308 (n=60)
  MACRO               : 0.6639

Accuracy: 0.6722
Macro F1: 0.6639


## 10. Confidence Threshold Tuning

In [12]:
print("\n🎯 Tuning confidence threshold...")

# Get validation predictions
val_pred = trainer.predict(tokenized_dataset['validation'])
val_logits = val_pred.predictions[0] if isinstance(val_pred.predictions, tuple) else val_pred.predictions
val_probs = torch.softmax(torch.tensor(val_logits), dim=-1).numpy()
val_max_probs = val_probs.max(axis=-1)
val_preds = val_logits.argmax(axis=-1)
val_labels = val_pred.label_ids

# Get "None" class ID
none_id = label_encoder.transform(['None'])[0]

# Tune threshold
best_tau, best_f1 = 0.0, 0.0
for tau in np.arange(0.30, 0.80, 0.05):
    # Apply threshold: low confidence → "None"
    adjusted = np.where(val_max_probs < tau, none_id, val_preds)
    f1 = f1_score(val_labels, adjusted, average='macro')
    if f1 > best_f1:
        best_f1, best_tau = f1, tau

print(f"\n✅ Optimal: τ={best_tau:.2f}, F1={best_f1:.4f}")

# Save
threshold_config = {'threshold': float(best_tau), 'f1_macro': float(best_f1), 'none_class_id': int(none_id)}
with open('models/roberta-optimized/optimal_threshold.json', 'w') as f:
    json.dump(threshold_config, f, indent=2)


🎯 Tuning confidence threshold...

📊 Per-class F1:
  Conflict            : 0.5625 (n=60)
  Economic            : 0.7939 (n=60)
  Human Impact        : 0.7031 (n=60)
  Moral Value         : 0.5091 (n=60)
  None                : 0.7840 (n=60)
  Powerlessness       : 0.6308 (n=60)
  MACRO               : 0.6639

✅ Optimal: τ=0.30, F1=0.6639


## 11. Test Set Evaluation

In [13]:
print("\n📊 Test set evaluation...")

test_pred = trainer.predict(tokenized_dataset['test'])
test_logits = test_pred.predictions[0] if isinstance(test_pred.predictions, tuple) else test_pred.predictions
test_probs = torch.softmax(torch.tensor(test_logits), dim=-1).numpy()
test_max_probs = test_probs.max(axis=-1)
test_preds_raw = test_logits.argmax(axis=-1)
test_labels = test_pred.label_ids

# Apply threshold
test_preds_final = np.where(test_max_probs < best_tau, none_id, test_preds_raw)

test_f1_raw = f1_score(test_labels, test_preds_raw, average='macro')
test_f1_final = f1_score(test_labels, test_preds_final, average='macro')

print(f"\nWithout threshold: {test_f1_raw:.4f}")
print(f"With threshold (τ={best_tau:.2f}): {test_f1_final:.4f}")

# Comparison
print(f"\nComparison:")
print(f"  BART baseline: 0.5873")
print(f"  BART optimized: 0.6238")
print(f"  RoBERTa+DAPT: {test_f1_final:.4f} ({((test_f1_final/0.5873-1)*100):+.1f}%)")


📊 Test set evaluation...



📊 Per-class F1:
  Conflict            : 0.5581 (n=58)
  Economic            : 0.6949 (n=58)
  Human Impact        : 0.7442 (n=58)
  Moral Value         : 0.6825 (n=58)
  None                : 0.7000 (n=58)
  Powerlessness       : 0.5983 (n=58)
  MACRO               : 0.6630

Without threshold: 0.6630
With threshold (τ=0.30): 0.6630

Comparison:
  BART baseline: 0.5873
  BART optimized: 0.6238
  RoBERTa+DAPT: 0.6630 (+12.9%)


## 12. Save Model

In [14]:
model.save_pretrained('models/roberta-optimized')
tokenizer.save_pretrained('models/roberta-optimized')
print("✅ Saved to models/roberta-optimized/")

results = {
    'train_metrics': train_result.metrics,
    'val_metrics': val_results,
    'test_f1_final': float(test_f1_final),
    'threshold': threshold_config,
}

with open('results/roberta_training_results.pkl', 'wb') as f:
    pickle.dump(results, f)
print("✅ Results saved")

✅ Saved to models/roberta-optimized/
✅ Results saved


## 13. Summary

In [15]:
print("\n" + "="*80)
print("ROBERTA TRAINING COMPLETE")
print("="*80)
print(f"\n🎯 Test Macro F1: {test_f1_final:.4f}")
print(f"   Improvement: {((test_f1_final/0.5873-1)*100):+.1f}% vs BART baseline")
print(f"\n💾 Saved:")
print(f"   - models/roberta-optimized/")
print(f"   - results/roberta_training_results.pkl")
print("="*80)


ROBERTA TRAINING COMPLETE

🎯 Test Macro F1: 0.6630
   Improvement: +12.9% vs BART baseline

💾 Saved:
   - models/roberta-optimized/
   - results/roberta_training_results.pkl
